In [ ]:
import numpy as np
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.ops import nearest_points
from shapely import MultiPoint
from config.config import DATA_RAW, DATA_INTERIM, OUTPUTS, CRS_BNG

## Control variables

Exploratory analysis and construction of control variables used in the regression
models, implementing the logic in `pipeline/05_build_controls.py`. Variables built:
drug offences (2024), IMD income and crime scores (2025), ethnic composition (2021
Census), TfL station distances, average property value, and population density.

## Drug offences

In [ ]:
london_lsoas = gpd.read_file(
    DATA_INTERIM / "boundaries" / "london_lsoa_2021.gpkg",
    layer="london_lsoa_2021"
)[["LSOA21CD", "LAD22NM"]]

In [ ]:
street_files = sorted((DATA_RAW / "met_police" / "street").glob("*.csv"))
street_df = pd.concat([pd.read_csv(f) for f in street_files], ignore_index=True)

print(f"Total rows: {len(street_df)}")


In [ ]:
# 1. Drop entirely null column
street_df = street_df.drop(columns=["Context"])

# 2. Drop rows with null LSOA code
null_lsoa = street_df["LSOA code"].isna().sum()
print(f"Dropping {null_lsoa} rows ({null_lsoa/len(street_df)*100:.2f}%) with null LSOA code")
street_df = street_df[street_df["LSOA code"].notna()].copy()

# 3. Filter to London LSOAs
street_df = street_df[street_df["LSOA code"].isin(london_lsoas["LSOA21CD"])].copy()
print(f"Rows after filtering to London: {len(street_df)}")

# 4. Now filter to drugs
drugs_df = street_df[street_df["Crime type"] == "Drugs"].copy()
print(f"Drug offence records: {len(drugs_df)}")

In [ ]:
drug_counts = (drugs_df.groupby("LSOA code")
                      .size()
                      .reset_index()
                      .rename(columns={"LSOA code": "LSOA21CD",
                                       0: "drug_count"}))

print(f"LSOAs with drug offences: {len(drug_counts)}")
print(f"LSOAs with no drug offences: "
      f"{len(london_lsoas) - len(drug_counts)}")
print(f"\nDrug offences per LSOA summary:")
print(drug_counts["drug_count"].describe())

In [ ]:
# Standardise by workplace population to match paper's drug offences rate
wp001_df = pd.read_csv(DATA_RAW / "census_2021" / "WP001_lsoa.csv")
wp001 = wp001_df[["Lower layer Super Output Areas Code", "Count"]].rename(columns={
    "Lower layer Super Output Areas Code": "LSOA21CD",
    "Count": "workplace_pop"
})

# Merge onto full LSOA list — fill zeros BEFORE computing rate
drug_offences_2024 = (london_lsoas[["LSOA21CD"]]
                      .merge(drug_counts, on="LSOA21CD", how="left")
                      .fillna({"drug_count": 0})
                      .merge(wp001, on="LSOA21CD", how="left"))

drug_offences_2024["drug_rate_2024"] = (
        drug_offences_2024["drug_count"] / drug_offences_2024["workplace_pop"]
)

print(f"Total LSOAs: {len(drug_offences_2024)}")
print(f"Null drug rates: {drug_offences_2024['drug_rate_2024'].isna().sum()}")
print(drug_offences_2024["drug_rate_2024"].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(drug_offences_2024["drug_count"],
             bins="auto", edgecolor="black", color="steelblue")
axes[0].set_title("Drug offences per LSOA (2024)")
axes[0].set_xlabel("Count")
axes[0].set_ylabel("Count of LSOAs")

axes[1].hist(np.log1p(drug_offences_2024["drug_count"]),
             bins="auto", edgecolor="black", color="steelblue")
axes[1].set_title("Drug offences per LSOA — log scale (2024)")
axes[1].set_xlabel("log(count + 1)")
axes[1].set_ylabel("Count of LSOAs")

plt.tight_layout()
plt.savefig(OUTPUTS / "figures" / "drug_offences_distribution.png",
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
os.makedirs(DATA_INTERIM / "controls", exist_ok=True)

drug_offences_2024[["LSOA21CD", "drug_rate_2024"]].to_csv(
    DATA_INTERIM / "controls" / "drug_offences_2024.csv",
    index=False
)

print(f"Saved drug offences rate for {len(drug_offences_2024)} LSOAs")

## Indices of Deprivation

Note: the London LSOA boundaries GeoPackage originally contained 22 duplicate
LSOA codes (44 rows), confirmed as exact row duplicates. This is fixed at source
in pipeline/03_build_lsoa_boundaries.py — the file loaded here contains 4,994
unique LSOAs.

In [ ]:
imd_df = pd.read_csv(
    DATA_RAW / "indices_deprivation" /
    "File_7_IoD2025_All_Ranks_Scores_Deciles_Population_Denominators.csv"
)

# Select and rename columns
imd_london = imd_df[[
    "LSOA code (2021)",
    "Income Score (rate)",
    "Crime Score"
]].rename(columns={
    "LSOA code (2021)": "LSOA21CD",
    "Income Score (rate)": "income_score",
    "Crime Score": "imd_crime_score"
})

# Filter to London
imd_london = imd_london[
    imd_london["LSOA21CD"].isin(london_lsoas["LSOA21CD"])
].copy()

print(f"London LSOAs in IMD data: {len(imd_london)}")
print(f"\nIncome score summary:")
print(imd_london["income_score"].describe())
print(f"\nCrime score summary:")
print(imd_london["imd_crime_score"].describe())
print(f"\nNull counts:")
print(imd_london.isna().sum())

In [ ]:
os.makedirs(DATA_INTERIM / "controls", exist_ok=True)

imd_london.to_csv(
    DATA_INTERIM / "controls" / "imd_2025.csv",
    index=False
)

print(f"Saved IMD scores for {len(imd_london)} LSOAs")

## Census 2021 - Ethnicity

In [ ]:
census_df = pd.read_csv(DATA_RAW / "census_2021" / "census2021-ts021-lsoa.csv")

# Calculate percentage non-white
census_london = census_df[["geography code",
                           "Ethnic group: Total: All usual residents",
                           "Ethnic group: White"]].rename(columns={
    "geography code": "LSOA21CD",
    "Ethnic group: Total: All usual residents": "total_residents",
    "Ethnic group: White": "white_residents"
})

# Filter to London
census_london = census_london[
    census_london["LSOA21CD"].isin(london_lsoas["LSOA21CD"])
].copy()

census_london["pct_non_white"] = (
        (census_london["total_residents"] - census_london["white_residents"])
        / census_london["total_residents"] * 100
)

print(f"London LSOAs: {len(census_london)}")
print(f"\nNull counts:\n{census_london.isna().sum()}")
print(f"\nPct non-white summary:")
print(census_london["pct_non_white"].describe())

In [ ]:
census_london[["LSOA21CD", "pct_non_white"]].to_csv(
    DATA_INTERIM / "controls" / "ethnic_composition_2021.csv",
    index=False
)

print(f"Saved ethnic composition for {len(census_london)} LSOAs")

## TfL station distances

In [ ]:
national_df = pd.read_csv(DATA_RAW / "tfl" / "Stops.csv", low_memory=False)

# Filter to relevant stop types and active only
stations_df = national_df[
    (national_df["StopType"].isin(["MET", "RLY", "PLT"])) &
    (national_df["Status"] == "active")
    ].copy()

print(f"Stations before geographic filter: {len(stations_df)}")
print(f"\nStop types:")
print(stations_df["StopType"].value_counts())

# Check coordinate columns
print(f"\nNull Easting: {stations_df['Easting'].isna().sum()}")
print(f"Null Northing: {stations_df['Northing'].isna().sum()}")
print(f"Null Longitude: {stations_df['Longitude'].isna().sum()}")

In [ ]:
# Create GeoDataFrame from stations
stations_gdf = gpd.GeoDataFrame(
    stations_df,
    geometry=gpd.points_from_xy(stations_df["Easting"], stations_df["Northing"]),
    crs=CRS_BNG
)

# Load London boundary — dissolve to single polygon for spatial filter
london_boundary = gpd.read_file(
    DATA_INTERIM / "boundaries" / "london_lsoa_2021.gpkg",
    layer="london_lsoa_2021"
).dissolve()

# Spatial join — keep only stations within London boundary
stations_london = gpd.sjoin(
    stations_gdf,
    london_boundary[["geometry"]],
    how="inner",
    predicate="within"
).drop(columns="index_right")

print(f"Stations within London boundary: {len(stations_london)}")
print(f"\nStop types:")
print(stations_london["StopType"].value_counts())
print(f"\nSample station names:")
print(stations_london["CommonName"].value_counts().head(20))

In [ ]:
# Remove tram stops and cable car stations
stations_london = stations_london[
    ~stations_london["CommonName"].str.contains("Tram Stop|IFS Cloud", na=False)
].copy()

# Deduplicate to one point per station using mean Easting/Northing
stations_unique = (stations_london
                   .groupby("CommonName")[["Easting", "Northing"]]
                   .mean()
                   .reset_index())

stations_unique = gpd.GeoDataFrame(
    stations_unique,
    geometry=gpd.points_from_xy(
        stations_unique["Easting"],
        stations_unique["Northing"]
    ),
    crs=CRS_BNG
)

print(f"Unique stations after filtering: {len(stations_unique)}")


In [ ]:
# Load S&S data
ss_df = pd.read_csv(DATA_INTERIM / "sands" / "ss_2025_london.csv")

ss_gdf = gpd.GeoDataFrame(
    ss_df,
    geometry=gpd.points_from_xy(ss_df["Longitude"], ss_df["Latitude"]),
    crs="EPSG:4326"
).to_crs(CRS_BNG)

print(f"S&S stops loaded: {len(ss_gdf)}")

# For each stop, calculate distance to nearest station
# Build a union geometry of all stations for efficient nearest neighbour lookup

stations_multipoint = MultiPoint(stations_unique.geometry.values)

ss_gdf["dist_to_tfl_m"] = ss_gdf.geometry.apply(
    lambda pt: pt.distance(nearest_points(stations_multipoint, pt)[0])
)

print(f"Distance calculation complete")
print(f"\nDistance summary (metres):")
print(ss_gdf["dist_to_tfl_m"].describe())

In [ ]:
tfl_dist_by_lsoa = (ss_gdf.groupby("LSOA21CD")["dist_to_tfl_m"]
                    .mean()
                    .reset_index()
                    .rename(columns={"dist_to_tfl_m": "mean_dist_to_tfl_m"}))

# Merge with full London LSOA list
tfl_dist_by_lsoa = (london_lsoas[["LSOA21CD"]]
                    .merge(tfl_dist_by_lsoa, on="LSOA21CD", how="left"))

print(f"Total LSOAs: {len(tfl_dist_by_lsoa)}")
print(f"LSOAs with no S&S stops (null distance): "
      f"{tfl_dist_by_lsoa['mean_dist_to_tfl_m'].isna().sum()}")
print(f"\nMean distance per LSOA summary (metres):")
print(tfl_dist_by_lsoa["mean_dist_to_tfl_m"].describe())

In [ ]:
tfl_dist_by_lsoa.to_csv(
    DATA_INTERIM / "controls" / "tfl_distances_2025.csv",
    index=False
)

print(f"Saved TfL distances for {tfl_dist_by_lsoa['mean_dist_to_tfl_m'].notna().sum()} LSOAs")

## Average property value

In [ ]:
land_reg_london = pd.read_csv(
    DATA_INTERIM / "land_registry" / "land_reg_london_2022_2024.csv"
)

print(f"Loaded {len(land_reg_london)} London transactions")

In [ ]:
avg_price_by_lsoa = (land_reg_london.groupby("lsoa21")["price"]
                     .mean()
                     .reset_index()
                     .rename(columns={"lsoa21": "LSOA21CD",
                                      "price": "mean_price"}))

# Merge with full London LSOA list
avg_price_by_lsoa = (london_lsoas[["LSOA21CD"]]
                     .merge(avg_price_by_lsoa, on="LSOA21CD", how="left"))

print(f"Total LSOAs: {len(avg_price_by_lsoa)}")
print(f"LSOAs with no transactions (null): "
      f"{avg_price_by_lsoa['mean_price'].isna().sum()}")
print(f"\nMean price summary:")
print(avg_price_by_lsoa["mean_price"].describe())

In [ ]:
avg_price_by_lsoa.to_csv(
    DATA_INTERIM / "controls" / "avg_property_value.csv",
    index=False
)

print(f"Saved average property values for "
      f"{avg_price_by_lsoa['mean_price'].notna().sum()} LSOAs")

## Census 2021 - Population Density

In [ ]:
# Load usual residents
ts001_df = pd.read_csv(DATA_RAW / "census_2021" / "census2021-ts001-lsoa.csv")
ts001_london = ts001_df[["geography code",
                         "Residence type: Total; measures: Value"]].rename(columns={
    "geography code": "LSOA21CD",
    "Residence type: Total; measures: Value": "usual_residents"
})
ts001_london = ts001_london[
    ts001_london["LSOA21CD"].isin(london_lsoas["LSOA21CD"])
].copy()

# Load workplace population
wp001_df = pd.read_csv(DATA_RAW / "census_2021" / "WP001_lsoa.csv")
wp001_london = wp001_df[["Lower layer Super Output Areas Code",
                         "Count"]].rename(columns={
    "Lower layer Super Output Areas Code": "LSOA21CD",
    "Count": "workplace_pop"
})
wp001_london = wp001_london[
    wp001_london["LSOA21CD"].isin(london_lsoas["LSOA21CD"])
].copy()

# Get LSOA area in hectares from boundaries GeoPackage
london_gdf = gpd.read_file(
    DATA_INTERIM / "boundaries" / "london_lsoa_2021.gpkg",
    layer="london_lsoa_2021"
)[["LSOA21CD", "geometry"]]
london_gdf["area_ha"] = london_gdf.geometry.area / 10000

print(f"Usual residents LSOAs: {len(ts001_london)}")
print(f"Workplace pop LSOAs: {len(wp001_london)}")
print(f"Boundary LSOAs: {len(london_gdf)}")

In [ ]:
pop_density = (london_lsoas[["LSOA21CD"]]
               .merge(ts001_london, on="LSOA21CD", how="left")
               .merge(wp001_london, on="LSOA21CD", how="left")
               .merge(london_gdf[["LSOA21CD", "area_ha"]], on="LSOA21CD", how="left"))

pop_density["pop_density"] = (
        (pop_density["usual_residents"] + pop_density["workplace_pop"])
        / pop_density["area_ha"]
)

print(f"Total LSOAs: {len(pop_density)}")
print(f"Null counts:\n{pop_density.isna().sum()}")
print(f"\nPopulation density summary (persons per hectare):")
print(pop_density["pop_density"].describe())

In [ ]:
pop_density[["LSOA21CD", "pop_density"]].to_csv(
    DATA_INTERIM / "controls" / "pop_density_2021.csv",
    index=False
)

print(f"Saved population density for {len(pop_density)} LSOAs")